---
format:
    html: default
    ipynb: default
jupyter: python3
---


## Explaining CNNs


In this assignment you implement two explainer algorithms for Convolutional Neural Networks (CNNs) and use them to inspect a model you have already trained. An explainer answers a question that accuracy alone cannot: which parts of an input drove this particular prediction? For an image classifier that answer takes the form of a saliency map, and a trustworthy map should highlight the animal rather than the background, a watermark, or some dataset artifact the model latched onto.

We invite you to watch the following video, which sets the context and walks through the tools you can use. Click the thumbnail to open it on YouTube.

[![Watch the assignment introduction on YouTube](https://img.youtube.com/vi/Am2EF9CLu-g/hqdefault.jpg)](https://www.youtube.com/watch?v=Am2EF9CLu-g)

In the [cats vs dogs](https://huggingface.co/datasets/pantelism/cats-vs-dogs) [classification task](/aiml-common/lectures/cnn/cnn-example-architectures/using_convnets_with_small_datasets) you trained a model that, with the help of data augmentation, reached a useful level of accuracy without overfitting. That trained model is your subject here. You do not retrain it or change its architecture; you attach explainers to it and interpret what they reveal about how it decides.

### The two methods you will implement

**Integrated gradients** ([Sundararajan et al., 2017](https://arxiv.org/abs/1703.01365)) attributes a prediction to individual input pixels. It picks a baseline image (commonly an all-black image, which the model should find uninformative) and integrates the gradient of the class score along the straight-line path from that baseline to the actual input. The result is a per-pixel attribution with two properties that plain input gradients lack: completeness, meaning the attributions sum to the difference in model output between the input and the baseline, and robustness to saturated activations, where a plain gradient would read close to zero even though the feature mattered.

**Grad-CAM** ([Selvaraju et al., 2016](https://arxiv.org/abs/1610.02391)) produces a coarse, class-discriminative heatmap. It takes the feature maps of a convolutional layer, usually the last one, and weights each map by the gradient of the class score flowing into it. Because it works at the resolution of a deep feature map rather than the raw pixels, it localizes the region the network used instead of scoring every pixel.

The two methods sit at opposite ends of a resolution trade-off: integrated gradients is fine-grained and pixel-level, Grad-CAM is coarse and region-level. Running both on the same image and noting where they agree, and where they do not, is the heart of this assignment.

We strongly advise PyTorch with [Captum](https://captum.ai/tutorials/) unless you already have Keras/TF expertise, because both methods ship as ready implementations there (`IntegratedGradients` and `LayerGradCam`). You are free to use the high-level APIs of the framework of your choice.

### What to submit

For each method, in the cells below:

- A markdown explanation written so that anyone who understands how a CNN works can follow it. Cover the intuition, the role of the baseline (integrated gradients) or the target layer (Grad-CAM), and one limitation of the method.
- Working code that runs the explainer on at least three correctly classified images and at least one image the model got wrong.
- The resulting maps overlaid on the input images, each with a one-line caption stating what the map suggests the model attended to.

### How your work is evaluated

- Correctness: the right baseline, the right target layer, and gradients taken with respect to the predicted class rather than a fixed label.
- Visualization quality: maps are overlaid on the inputs, readable, and labeled.
- Depth of interpretation: noting that a map "looks reasonable" is not enough. Point to where the two methods agree, where they disagree, and what the misclassified example tells you about what the model actually learned.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from captum.attr import IntegratedGradients, LayerAttribution, LayerGradCam
from datasets import load_dataset
from PIL import UnidentifiedImageError
from torchvision import transforms


IMG_SIZE = 150
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)


def resolve_model_path():
    candidates = [
        Path("notebooks/cnn/cats_and_dogs_small.pth"),
        Path("../../notebooks/cnn/cats_and_dogs_small.pth"),
        Path("../notebooks/cnn/cats_and_dogs_small.pth"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find cats_and_dogs_small.pth")


MODEL_PATH = resolve_model_path()
print(f"Device: {DEVICE}")
print(f"Model path: {MODEL_PATH}")


## Load the trained model and test data

I reuse the same CNN architecture and the same preprocessing from the training notebook. That matters because the saved weights only make sense if the architecture and normalization match what the model saw during training.


In [ ]:
class SmallConvNet(nn.Module):
    def __init__(self, dropout=True):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5) if dropout else nn.Identity(),
            nn.Linear(128 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x).squeeze(1)


basic_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


model = SmallConvNet(dropout=True)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()


ds_dict = load_dataset("pantelism/cats-vs-dogs")
label_names = ds_dict["test"].features["label"].names
test_split = ds_dict["test"]

print("Labels:", label_names)
print("Test size:", len(test_split))


In [ ]:
def prepare_example(hf_split, idx):
    while True:
        try:
            sample = hf_split[idx]
            image = sample["image"].convert("RGB")
            label = int(sample["label"])
            tensor = basic_tf(image)
            return image, tensor, label
        except UnidentifiedImageError:
            idx = (idx + 1) % len(hf_split)


def predict_single(model, input_tensor):
    with torch.no_grad():
        logits = model(input_tensor.unsqueeze(0).to(DEVICE))
        dog_prob = torch.sigmoid(logits)[0].item()
        pred_label = int(dog_prob > 0.5)
        pred_prob = dog_prob if pred_label == 1 else 1.0 - dog_prob
    return pred_label, pred_prob


def collect_examples(model, hf_split, n_correct=3, n_wrong=1):
    examples = []
    correct = 0
    wrong = 0

    for idx in range(len(hf_split)):
        image, tensor, true_label = prepare_example(hf_split, idx)
        pred_label, pred_prob = predict_single(model, tensor)
        is_correct = pred_label == true_label

        if is_correct and correct < n_correct:
            examples.append({
                "index": idx,
                "image": image,
                "tensor": tensor,
                "true_label": true_label,
                "pred_label": pred_label,
                "pred_prob": pred_prob,
                "correct": True,
            })
            correct += 1
        elif (not is_correct) and wrong < n_wrong:
            examples.append({
                "index": idx,
                "image": image,
                "tensor": tensor,
                "true_label": true_label,
                "pred_label": pred_label,
                "pred_prob": pred_prob,
                "correct": False,
            })
            wrong += 1

        if correct >= n_correct and wrong >= n_wrong:
            break

    if correct < n_correct or wrong < n_wrong:
        raise RuntimeError("Could not find enough correct and incorrect examples")

    return examples


examples = collect_examples(model, test_split, n_correct=3, n_wrong=1)
print(f"Collected {len(examples)} examples")
for ex in examples:
    outcome = "correct" if ex["correct"] else "misclassified"
    print(
        f"idx={ex['index']} | true={label_names[ex['true_label']]} | "
        f"pred={label_names[ex['pred_label']]} | conf={ex['pred_prob']:.3f} | {outcome}"
    )


In [ ]:
def denormalize_image(tensor):
    image = tensor.detach().cpu().clone()
    image = image * 0.5 + 0.5
    image = image.clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


def normalize_heatmap(heatmap):
    heatmap = heatmap.astype(np.float32)
    heatmap = heatmap - heatmap.min()
    max_value = heatmap.max()
    if max_value > 0:
        heatmap = heatmap / max_value
    return heatmap


def overlay_heatmap(image, heatmap, alpha=0.45, cmap_name="jet"):
    cmap = plt.get_cmap(cmap_name)
    color_heatmap = cmap(heatmap)[..., :3]
    blended = (1 - alpha) * image + alpha * color_heatmap
    return np.clip(blended, 0, 1)


def describe_attention_region(heatmap):
    h, w = heatmap.shape
    total = heatmap.sum()
    if total <= 1e-8:
        return "the map is weak, so there is no clear hotspot"

    threshold = 0.6 * heatmap.max()
    mask = heatmap >= threshold
    if mask.sum() == 0:
        mask = heatmap >= heatmap.max()

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()

    box_h = (y1 - y0 + 1) / h
    box_w = (x1 - x0 + 1) / w
    area = box_h * box_w

    cy = ((y0 + y1) / 2) / h
    cx = ((x0 + x1) / 2) / w

    if cy < 0.35:
        vertical = "upper"
    elif cy > 0.65:
        vertical = "lower"
    else:
        vertical = "middle"

    if cx < 0.35:
        horizontal = "left"
    elif cx > 0.65:
        horizontal = "right"
    else:
        horizontal = "center"

    if area < 0.12:
        spread = "a compact hotspot"
    elif area < 0.28:
        spread = "a medium-sized hotspot"
    else:
        spread = "a broad hotspot"

    if box_w > 1.4 * box_h:
        shape = "that stretches more horizontally"
    elif box_h > 1.4 * box_w:
        shape = "that stretches more vertically"
    else:
        shape = "with a fairly balanced shape"

    if vertical == "middle" and horizontal == "center":
        location = "near the center"
    elif vertical == "middle":
        location = f"along the {horizontal} side"
    elif horizontal == "center":
        location = f"in the {vertical} part of the image"
    else:
        location = f"in the {vertical}-{horizontal} part of the image"

    return f"the strongest response is {spread} {location}, {shape}"


fig, axes = plt.subplots(len(examples), 1, figsize=(4, 4 * len(examples)))
if len(examples) == 1:
    axes = [axes]

for ax, ex in zip(axes, examples):
    ax.imshow(denormalize_image(ex["tensor"]))
    outcome = "correct" if ex["correct"] else "misclassified"
    ax.set_title(
        f"idx {ex['index']} | true={label_names[ex['true_label']]} | "
        f"pred={label_names[ex['pred_label']]} ({ex['pred_prob']:.3f}) | {outcome}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


## Integrated Gradients

Integrated Gradients is meant to show which pixels helped the model make its prediction. A plain gradient only looks at the image at one point, and that can be noisy or close to zero. Integrated Gradients avoids that by starting from a baseline image and moving toward the real image in small steps. At each step it measures how the predicted class score changes, then adds those changes together.

In this notebook I use an **all-black baseline**. Because the training images were normalized, the baseline tensor is filled with `-1`, which matches a black image before normalization. The baseline matters because the attributions are measuring how the prediction changes from that starting point to the real image.

One limitation is that the result depends on the baseline. If I changed the baseline image, the attribution map could also change.


In [ ]:
def predicted_class_score(inputs, predicted_class):
    logits = model(inputs)
    dog_prob = torch.sigmoid(logits)
    return torch.where(predicted_class == 1, dog_prob, 1 - dog_prob)


ig = IntegratedGradients(predicted_class_score)


def integrated_gradients_heatmap(example, steps=64):
    input_tensor = example["tensor"].unsqueeze(0).to(DEVICE)
    baseline = torch.full_like(input_tensor, -1.0)
    pred_class = torch.tensor([example["pred_label"]], device=DEVICE)

    attributions = ig.attribute(
        input_tensor,
        baselines=baseline,
        additional_forward_args=(pred_class,),
        n_steps=steps,
    )

    heatmap = attributions.abs().sum(dim=1)[0].detach().cpu().numpy()
    return normalize_heatmap(heatmap)


ig_maps = []
for ex in examples:
    heatmap = integrated_gradients_heatmap(ex)
    ig_maps.append(heatmap)


fig, axes = plt.subplots(len(examples), 2, figsize=(10, 4 * len(examples)))
if len(examples) == 1:
    axes = np.array([axes])

for row, ex in enumerate(examples):
    image = denormalize_image(ex["tensor"])
    heatmap = ig_maps[row]
    overlay = overlay_heatmap(image, heatmap)
    caption = describe_attention_region(heatmap)

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(
        f"Original | true={label_names[ex['true_label']]} | pred={label_names[ex['pred_label']]}"
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title("Integrated Gradients")
    axes[row, 1].set_xlabel(caption)
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()


## Grad-CAM

Grad-CAM looks at the problem at the feature-map level instead of the pixel level. It asks which region of the image mattered most for the predicted class. To do that, it takes the gradients flowing into a convolutional layer and uses them to weight that layer's feature maps. The final result is a coarse heatmap showing the region the network used.

I use the **last convolutional layer** as the target layer: `model.features[9]`. This is a good choice because it still keeps spatial information, but it is also close to the output, so it reflects the model's final decision better than an earlier layer would.

One limitation is that Grad-CAM is low-resolution. It is useful for showing the general area the model focused on, but it does not give precise pixel-level detail.


In [ ]:
gradcam = LayerGradCam(predicted_class_score, model.features[9])


def gradcam_heatmap(example):
    input_tensor = example["tensor"].unsqueeze(0).to(DEVICE)
    pred_class = torch.tensor([example["pred_label"]], device=DEVICE)

    cam = gradcam.attribute(input_tensor, additional_forward_args=(pred_class,))
    cam = LayerAttribution.interpolate(cam, (IMG_SIZE, IMG_SIZE))
    cam = torch.relu(cam)
    heatmap = cam[0, 0].detach().cpu().numpy()
    return normalize_heatmap(heatmap)


gradcam_maps = []
for ex in examples:
    heatmap = gradcam_heatmap(ex)
    gradcam_maps.append(heatmap)


fig, axes = plt.subplots(len(examples), 2, figsize=(10, 4 * len(examples)))
if len(examples) == 1:
    axes = np.array([axes])

for row, ex in enumerate(examples):
    image = denormalize_image(ex["tensor"])
    heatmap = gradcam_maps[row]
    overlay = overlay_heatmap(image, heatmap)
    caption = describe_attention_region(heatmap)

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(
        f"Original | true={label_names[ex['true_label']]} | pred={label_names[ex['pred_label']]}"
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title("Grad-CAM")
    axes[row, 1].set_xlabel(caption)
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()


## Side-by-side comparison

The main point of the assignment is to compare the two methods on the same images. Integrated Gradients gives a more detailed map, while Grad-CAM gives a rougher region-based map. Looking at both together makes it easier to judge whether the model is focusing on the animal itself or on something around it.


In [ ]:
fig, axes = plt.subplots(len(examples), 3, figsize=(14, 4 * len(examples)))
if len(examples) == 1:
    axes = np.array([axes])

for row, ex in enumerate(examples):
    image = denormalize_image(ex["tensor"])
    ig_heatmap = ig_maps[row]
    gc_heatmap = gradcam_maps[row]

    ig_overlay = overlay_heatmap(image, ig_heatmap)
    gc_overlay = overlay_heatmap(image, gc_heatmap)

    ig_caption = describe_attention_region(ig_heatmap)
    gc_caption = describe_attention_region(gc_heatmap)

    agreement = "agree" if ig_caption == gc_caption else "disagree"
    outcome = "correct" if ex["correct"] else "misclassified"

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(
        f"idx {ex['index']} | true={label_names[ex['true_label']]} | "
        f"pred={label_names[ex['pred_label']]} ({ex['pred_prob']:.3f}) | {outcome}"
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(ig_overlay)
    axes[row, 1].set_title("Integrated Gradients")
    axes[row, 1].set_xlabel(ig_caption)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(gc_overlay)
    axes[row, 2].set_title("Grad-CAM")
    axes[row, 2].set_xlabel(gc_caption)
    axes[row, 2].axis("off")

    print(
        f"Example {row + 1}: {outcome}. "
        f"Integrated Gradients says {ig_caption}; Grad-CAM says {gc_caption}. "
        f"The two methods {agreement} on the broad attention region."
    )

plt.tight_layout()
plt.show()


## Discussion

Looking at the explanation maps, Integrated Gradients and Grad-CAM sometimes agree on the main subject of the image, but they do not always highlight it in the same way. Integrated Gradients usually gives a more detailed map and often focuses on smaller parts of the cat, especially around the face. Grad-CAM is much coarser and tends to highlight larger regions instead of individual features.

For the correctly classified examples, the model usually seems to focus on the cat itself. In Example 1 (`true=cat`, `pred=cat`, confidence `0.922`), Grad-CAM highlights most of the cat very strongly, while Integrated Gradients is flatter and less concentrated. In Example 3 (`true=cat`, `pred=cat`, confidence `0.805`), Integrated Gradients places stronger emphasis on the cat's face, ears, and paws, while Grad-CAM lights up most of the image except the lower corners. In Example 4 (`true=cat`, `pred=cat`, confidence `0.868`), Integrated Gradients again focuses most clearly on the face, especially around the eyes, nose, and mouth, while Grad-CAM highlights a broader vertical region covering the cat from the nose downward. These correct examples suggest that the model often relies on facial features, but Grad-CAM shows that it is also using larger surrounding regions.

The misclassified example is the most revealing one. In Example 2 (`true=cat`, `pred=dog`, confidence `0.922`), Integrated Gradients still places attention on parts of the cat, especially around its outline, while Grad-CAM highlights the face region but also gives strong attention to the bottom-right corner where the name "Felix" appears. That suggests the model may not be relying only on the animal itself. It may also be using text or other image artifacts as cues, which could push it toward the wrong class. This is exactly the kind of behavior that explanation methods are useful for uncovering.

Overall, Integrated Gradients was more useful for showing fine-grained details such as the face, ears, paws, and outline of the cat. Grad-CAM was more useful for showing the broader region the model relied on. When both methods emphasize the cat's face or body, the prediction seems more trustworthy. When the methods disagree strongly, or when attention spreads onto text or other non-animal regions, it suggests the model may have learned shortcuts instead of focusing only on the cat itself.
